# Lab 10: GPT Generation

**Module 10** | Read `notes/10-gpt-causal-lms.pdf` before running this notebook.

Load a small causal language model and compare greedy decoding, temperature sampling, and nucleus (top-p) sampling.

## How to use this notebook

1. **Run cells top to bottom** (Shift+Enter). Do not skip markdown cells; they explain the code.
2. **Read before you run.** Each section tells you what the next code block does, which terms mean, and what the printed output should look like.
3. **Use the comments in code.** Lines starting with `#` explain what each step does in plain language.
4. **Check the output.** After each code cell, compare your printed numbers to the "What to look for" notes.

Code is complete. You are not expected to fill in missing sections or debug skeleton code.


## Runtime device (CPU or GPU)

**What this section does:** PyTorch can run math on your **CPU** (the main processor) or on an **NVIDIA GPU** through **CUDA**. GPUs are faster for neural networks because they run many small matrix operations in parallel.

**Key terms:**
- **CPU:** Always available. Training works but can be slower on large models.
- **GPU / CUDA:** Optional hardware acceleration. If you have an NVIDIA GPU and drivers installed, PyTorch will use it automatically.
- **VRAM:** GPU memory. If you run out, reduce batch size.

The next cell prints which device is active so you know what to expect for training speed.


In [ ]:
import torch

# Pick GPU if CUDA is available; otherwise fall back to CPU (labs still run).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("Using CPU (no CUDA GPU detected).")
    print("All labs still run; training may take longer than on a GPU.")


## Lab 10: GPT-2 text generation

**GPT-2** is a causal language model: it reads tokens left to right and predicts the next token. At inference time we **decode** text one token at a time. This lab compares **greedy decoding** with **sampling** using **temperature** and **top-p (nucleus) sampling**.


## Key terms for this lab

| Term | Meaning |
|------|---------|
| **Causal language model** | A model that predicts the next token using only previous tokens (no peeking ahead). |
| **Token** | A text piece (word or subword) represented as an integer ID in the model vocabulary. |
| **Decoding** | The process of choosing tokens one by one to extend a prompt into longer text. |
| **Greedy decoding** | Always pick the single highest-probability next token. Deterministic and often repetitive. |
| **Sampling** | Randomly draw the next token from the probability distribution instead of always taking the top one. |
| **Logits** | Raw scores for each vocabulary token before converting to probabilities. |
| **Softmax** | Turns logits into probabilities that sum to 1 across the vocabulary. |
| **Temperature** | A number that scales logits before softmax. Lower = more confident/peaked; higher = flatter/more random. |
| **Top-p (nucleus sampling)** | Keep the smallest set of tokens whose cumulative probability exceeds p, renormalize, then sample. |
| **do_sample** | Hugging Face flag: False = greedy argmax; True = stochastic sampling with temperature/top_p. |
| **Prompt** | The starting text you provide. The model continues from there. |
| **Autoregressive** | Each new token is fed back as input for predicting the following token. |

Refer back to this table whenever a term appears in code or output.


### Concept: Greedy decoding vs sampling

**Greedy decoding** (`do_sample=False`): At each step, compute probabilities for every next token, then **always** choose the token with the highest probability (the **argmax**). If you run the same prompt twice, you get the **same** output. Greedy is predictable but can loop ("the the the") or sound dull because it never takes slightly less likely but more interesting words.

**Sampling** (`do_sample=True`): At each step, treat the probability distribution like a weighted lottery. Tokens with higher probability are **more likely** to be picked, but not guaranteed. You can get different continuations each run. Sampling produces more varied, creative text at the cost of occasional surprises or mistakes.

**Why we use `do_sample=True` for creative generation:** Real writing is not always the single safest next word. Sampling explores plausible alternatives. For factual or code completion tasks you might prefer greedy or low temperature; for brainstorming or story continuation, sampling with temperature and top-p is standard.


### Concept: What temperature does to the probability distribution

Before softmax, the model outputs logits `z_i` for each token `i`. Temperature `T` scales them: `z_i / T`.

- **Low temperature (e.g., 0.3):** Dividing by a small T **magnifies** differences between logits. After softmax, one token dominates (e.g., 90% probability). Sampling behaves almost greedy but still allows rare picks.
- **Temperature 1.0:** Uses the model's raw preferences unchanged.
- **High temperature (e.g., 1.2):** Dividing by a large T **shrinks** differences. Probabilities spread out; unlikely tokens gain chance. Output becomes more surprising and sometimes incoherent.

Temperature does **not** change what the model learned; it only changes how sharply we read its next-token preferences during decoding.


### Concept: Top-p (nucleus) sampling step by step

**Goal:** Avoid picking from the entire vocabulary (50,000+ tokens for GPT-2). Many tokens are extremely unlikely ("zebra" after "The future of machine learning is"). Top-p focuses on the **smallest** high-probability group.

**Steps at each generated token:**

1. Compute logits for all vocabulary tokens; apply temperature scaling.
2. Softmax to probabilities `p_i`, sorted largest to smallest.
3. Walk down the sorted list, accumulating cumulative probability until the sum **exceeds p** (here 0.9). That set is the **nucleus**.
4. Zero out every token **outside** the nucleus.
5. Renormalize the remaining probabilities so they sum to 1.
6. **Sample** one token from this renormalized distribution.
7. Append the token to the sequence and repeat.

**Intuition:** With `top_p=0.9`, you ignore the long tail of bizarre tokens but still allow several reasonable continuations. Combined with temperature, you control both **how peaked** the distribution is and **how many** candidates you consider.


### Step 1: Load GPT-2 and print model info

**What this does:** Loads GPT-2 (or DistilGPT-2 fallback), moves to `device`, and prints architecture statistics plus the prompt.

**Why we do this:** Confirms the model fits in memory and sets shared constants (`PROMPT`, `MAX_NEW_TOKENS`) for later decoding cells.


**What to look for in the output**

Model name gpt2 or distilgpt2. Vocab size 50257 for GPT-2. Parameters around 124M. Prompt and max new tokens printed at the end.


In [ ]:
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

ROOT = Path("..").resolve()
PROMPT = "The future of machine learning is"
TEMPERATURES = [0.3, 0.7, 1.2]  # low, medium, high randomness
TOP_P = 0.9  # nucleus threshold
MAX_NEW_TOKENS = 60  # how many new tokens to generate after the prompt

try:
    model_name = "gpt2"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
except Exception:
    # Smaller fallback if download or memory fails
    model_name = "distilgpt2"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token  # GPT-2 has no pad token by default
model.to(device)
model.eval()

vocab_size = tokenizer.vocab_size
total_params = sum(p.numel() for p in model.parameters())
config = model.config

print("Model info")
print(f"  Name:              {model_name}")
print(f"  Device:            {device}")
print(f"  Vocab size:        {vocab_size:,}")
print(f"  Parameters:        {total_params:,}")
print(f"  Hidden size:       {config.n_embd}")
print(f"  Layers:            {config.n_layer}")
print(f"  Attention heads:   {config.n_head}")
print(f"  Max context:       {config.n_positions} tokens")
print(f"\nPrompt: {PROMPT!r}")
print(f"Max new tokens: {MAX_NEW_TOKENS}")


### Step 2: Greedy decoding baseline

**What this does:** Generates text with `do_sample=False`, which forces argmax (greedy) decoding every step.

**Why we do this:** Greedy output is the deterministic baseline. You will compare its repetition and tone against sampled outputs in later steps.


**What to look for in the output**

New tokens generated should be 60 (or close if EOS stops early). Full text starts with the prompt and continues deterministically. Same result every rerun.


In [ ]:
inputs = tokenizer(PROMPT, return_tensors="pt").to(device)
prompt_len = inputs["input_ids"].shape[1]

with torch.no_grad():
    greedy_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,  # greedy: always pick highest-probability token
        pad_token_id=tokenizer.eos_token_id,
    )

greedy_text = tokenizer.decode(greedy_ids[0], skip_special_tokens=True)
greedy_new_tokens = greedy_ids.shape[1] - prompt_len

print("Greedy decoding (deterministic argmax)")
print(f"  New tokens generated: {greedy_new_tokens}")
print(f"  Full text:\n{greedy_text}")


### Step 3: Nucleus sampling at three temperatures

**What this does:** For each temperature, runs `do_sample=True` with `top_p=0.9` and a fixed random seed, prints full text, and builds a summary table.

**Why we do this:** Same prompt and seed isolate the effect of temperature. You should see low T stay focused, high T wander more.


**What to look for in the output**

Three blocks of generated text with headers showing temperature and top_p. Texts should differ across temperatures. Summary table lists temperature, top_p, new_tokens.


In [ ]:
rows = []

for temp in TEMPERATURES:
    # Fixed seed for fair comparison across temperatures
    torch.manual_seed(42)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(42)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,  # enable stochastic sampling (required for temperature/top_p)
            temperature=temp,  # sharpen (low) or flatten (high) the distribution
            top_p=TOP_P,  # nucleus sampling threshold
            pad_token_id=tokenizer.eos_token_id,
        )

    text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    new_tokens = output_ids.shape[1] - prompt_len
    rows.append({
        "method": f"sample (T={temp}, top_p={TOP_P})",
        "temperature": temp,
        "top_p": TOP_P,
        "new_tokens": new_tokens,
        "generated_text": text,
    })

    print(f"\n{'=' * 60}")
    print(f"Temperature = {temp} | top_p = {TOP_P} | new tokens = {new_tokens}")
    print(f"{'=' * 60}")
    print(text)

df = pd.DataFrame(rows)
print("\nSummary table")
print(df[["temperature", "top_p", "new_tokens"]].to_string(index=False))


### Step 4: Compare decoding strategies

**What this does:** Combines greedy output with the three sampled runs into one comparison table and prints each full text.

**Why we do this:** Side-by-side reading highlights repetition (greedy), focus (low T), and diversity (high T). Top-p prevented any run from sampling extremely rare tokens.


**What to look for in the output**

Four sections: greedy plus three sampled methods. Compare repetition, specificity, and tone. Takeaway line printed at the end.


In [ ]:
comparison = [
    {
        "method": "greedy (argmax)",
        "temperature": 0.0,
        "top_p": 1.0,
        "new_tokens": greedy_new_tokens,
        "generated_text": greedy_text,
    },
    *rows,
]

compare_df = pd.DataFrame(comparison)
print("All decoding runs")
for i, row in compare_df.iterrows():
    print(f"\n--- {row['method']} ({row['new_tokens']} new tokens) ---")
    print(row["generated_text"])

print("\nTakeaway: compare repetition, specificity, and tone across methods.")
